# Week 16 — Train and Sample a Tiny DDPM on MNIST

Use the [working copy](w16_train_tiny_ddpm_mnist_solved.ipynb) for your exercises. Both notebooks start with the same cells.

This builds on the [Week 15 forward process](../notes/w15_forward_process_discrete.md) and [Week 16 discussion](../notes/w16_hf_ddpm_training.md). Day 4 covers the training batch, one optimizer update, and a fixed-batch check. Day 5 covers reverse sampling and the full experiment.

## Setup

Reuse the Week 11 MNIST training cache when available. Select a fixed subset and normalize images to $[-1,1]$ as in Week 15. The starting configuration runs on CPU.

In [ ]:
from pathlib import Path
import gzip
import hashlib
import os
import struct
import urllib.request

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
CACHE = ROOT / "scratch" / "w16_train_tiny_ddpm_mnist"
CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE / "matplotlib"))
import matplotlib.pyplot as plt

SEED = 16
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(torch.get_num_threads(), 4))
config = {
    "seed": SEED,
    "T": 1000,
    "beta_start": 1e-4,
    "beta_end": 0.02,
    "n_train": 5000,
    "batch_size": 64,
    "channels": 16,
    "learning_rate": 1e-3,
    "epochs": 2,
    "grid_count": 16,
}
assert 0 < config["beta_start"] <= config["beta_end"] < 1
print(f"CPU | seed={SEED} | T={config['T']}")

In [ ]:
def load_mnist_images() -> np.ndarray:
    """Return the verified MNIST training images as uint8."""
    filename = "train-images-idx3-ubyte.gz"
    checksum = "f68b3c2dcbeaaa9fbdd348bbdeb94873"
    previous = ROOT / "scratch" / "w11_vae" / "mnist" / filename
    target = previous if previous.is_file() else CACHE / filename
    if target.is_file():
        payload = target.read_bytes()
    else:
        url = "https://ossci-datasets.s3.amazonaws.com/mnist/" + filename
        with urllib.request.urlopen(url, timeout=60) as response:
            payload = response.read()
    if hashlib.md5(payload).hexdigest() != checksum:
        raise ValueError(f"MNIST image checksum mismatch for {target}")
    if not target.is_file():
        target.write_bytes(payload)
    raw = gzip.decompress(payload)
    assert struct.unpack(">IIII", raw[:16]) == (2051, 60000, 28, 28)
    return np.frombuffer(raw, dtype=np.uint8, offset=16).reshape(-1, 28, 28)


images = load_mnist_images()
indices = rng.choice(len(images), config["n_train"], replace=False)
train_x = torch.from_numpy(images[indices, None]).float() / 127.5 - 1.0


def make_train_loader() -> DataLoader:
    """Return a loader with reproducible minibatch order."""
    return DataLoader(
        TensorDataset(train_x),
        batch_size=config["batch_size"],
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )


(clean_batch,) = next(iter(make_train_loader()))
assert clean_batch.shape == (config["batch_size"], 1, 28, 28)
assert clean_batch.min() >= -1 and clean_batch.max() <= 1

fig, axes = plt.subplots(1, 4, figsize=(5, 1.5))
for axis, image in zip(axes, clean_batch[:4]):
    axis.imshow(image[0], cmap="gray", vmin=-1, vmax=1)
    axis.axis("off")
fig.tight_layout()
plt.show()

## Step 1: Schedule and model

Schedule tables use mathematical indices $0,\ldots,T$. The entries $\beta_0=0$ and $\bar\alpha_0=1$ represent the clean boundary. Training draws timesteps from $1,\ldots,T$.

In [ ]:
T = config["T"]
betas = torch.cat((torch.zeros(1), torch.linspace(config["beta_start"], config["beta_end"], T)))
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)


def coefficient_at(table: torch.Tensor, t: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
    """Gather one coefficient per image and broadcast over its coordinates."""
    assert t.dtype == torch.long and t.shape == (batch.shape[0],)
    return table[t].reshape(batch.shape[0], 1, 1, 1)


assert alpha_bar.shape == (T + 1,) and alpha_bar[0] == 1
assert torch.all(alpha_bar[1:] < alpha_bar[:-1])
times = torch.tensor([1, T])
assert coefficient_at(alpha_bar, times, clean_batch[:2]).shape == (2, 1, 1, 1)

Follow the timestep through the model. A small network turns the scaled timestep into one value per hidden channel for each image. Broadcasting lets us add those values to the first convolution's feature map. The final convolution predicts one noise value per pixel.

In [ ]:
class TinyDenoiser(nn.Module):
    """Predict image-shaped noise from a noisy image and its timestep."""

    def __init__(self, timesteps: int, channels: int):
        super().__init__()
        self.timesteps = timesteps
        self.input_conv = nn.Conv2d(1, channels, kernel_size=3, padding=1)
        self.time_projection = nn.Sequential(
            nn.Linear(1, channels),
            nn.SiLU(),
            nn.Linear(channels, channels),
        )
        self.output_layers = nn.Sequential(
            nn.SiLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=2, dilation=2),
            nn.SiLU(),
            nn.Conv2d(channels, 1, kernel_size=3, padding=1),
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_scaled = t.to(dtype=x_t.dtype).reshape(-1, 1) / self.timesteps
        time_features = self.time_projection(t_scaled)[:, :, None, None]
        return self.output_layers(self.input_conv(x_t) + time_features)


model = TinyDenoiser(T, config["channels"])
with torch.no_grad():
    prediction = model(clean_batch[:2], times)
    assert prediction.shape == clean_batch[:2].shape and torch.isfinite(prediction).all()
print(f"Parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

## Exercise: Construct the training batch and update the weights

### TODO 1 — Construct one batched training example

Use the supplied generator. Decide what to draw and which shapes you need, then return the noisy images, their timesteps, and the noise target. The forward equation and the schedule lookup are available as references.

In [ ]:
def prepare_batch(
    clean: torch.Tensor, generator: torch.Generator
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return (noisy images, timesteps, target noise) for a clean batch."""
    raise NotImplementedError("TODO 1: construct a noisy training batch.")

### TODO 2 — Perform one noise-MSE optimizer update

Use the optimizer and corrupted batch passed in. Return the loss for the update as a Python float. Keep the latest gradients available for inspection after the update.

A common pitfall is forgetting that gradients accumulate across backward calls.

In [ ]:
def train_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    x_t: torch.Tensor,
    t: torch.Tensor,
    noise: torch.Tensor,
) -> float:
    """Perform one optimizer update and return its noise-MSE loss."""
    raise NotImplementedError("TODO 2: implement one training update.")

## Step 2: Check the result on one fixed corruption

Run this cell after filling both exercises. It uses a separate model and holds the inputs and targets fixed for 20 updates. Compare initial and final loss on that same example. Day 5 will draw a new corruption for each training batch.

Optional extension. Try a different fixed corruption and compare the losses.

In [ ]:
check_rng = torch.Generator().manual_seed(SEED + 1)
check_x0 = clean_batch[:16]
x_t, t, noise = prepare_batch(check_x0, check_rng)
assert x_t.shape == noise.shape == check_x0.shape
assert t.shape == (len(check_x0),) and t.dtype == torch.long
assert torch.all((1 <= t) & (t <= T))
assert torch.isfinite(x_t).all() and torch.isfinite(noise).all()
for i in (0, len(check_x0) - 1):
    weight = alpha_bar[t[i]]
    expected = weight.sqrt() * check_x0[i] + (1 - weight).sqrt() * noise[i]
    torch.testing.assert_close(x_t[i], expected)

torch.manual_seed(SEED)
check_model = TinyDenoiser(T, config["channels"])
optimizer = torch.optim.Adam(check_model.parameters(), lr=config["learning_rate"])
before = [parameter.detach().clone() for parameter in check_model.parameters()]
with torch.no_grad():
    initial_loss = nn.functional.mse_loss(check_model(x_t, t), noise).item()

losses = [train_step(check_model, optimizer, x_t, t, noise) for _ in range(20)]
assert np.isfinite(losses).all() and np.isfinite(initial_loss)
assert all(
    parameter.grad is not None and torch.isfinite(parameter.grad).all()
    for parameter in check_model.parameters()
)
assert any(not torch.equal(old, new) for old, new in zip(before, check_model.parameters()))
with torch.no_grad():
    final_loss = nn.functional.mse_loss(check_model(x_t, t), noise).item()
assert np.isfinite(final_loss)
print(f"Fixed batch loss: {initial_loss:.4f} to {final_loss:.4f}")
plt.plot(range(len(losses) + 1), [*losses, final_loss])
plt.xlabel("Optimizer updates")
plt.ylabel("Noise MSE")
plt.show()

## Wrap-up

Record whether the loss improved and what that shows about the training path. The first generated images will provide a separate check in Day 5. Start that session from the [reverse-loop pseudocode](../notes/w16_hf_ddpm_training.md#day-3--sampling-from-a-trained-ddpm).